# Matrix Experimentation Walkthrough
This notebook demonstrates how to use the library to generate, sparsify, and visualize adjacency matrices for time series data, and finally divide nodes into batches.


## 1. Setup and Imports


In [ ]:
import sys
import os
import numpy as np
import matplotlib.pyplot as plt

# Standard way to add project root to path
project_root = os.path.abspath('..')
if project_root not in sys.path:
    sys.path.insert(0, project_root)

from src.matrix_generation import PearsonCorrelationGenerator
from src.sparcification import MatrixConstructionPipeline, TopKRowSparsifier, RowL1Normalizer
from src.utils.visualization import plot_adjacency_matrix_heatmap, plot_node_time_series
from src.batching import KHopBatcher

print("Setup Complete.")


ModuleNotFoundError: No module named 'matrix_generation'

## 2. Load Data
We'll use a synthetic dataset.


In [ ]:
from src.data.loaders import load_gaussian_data
data, _ = load_gaussian_data(n_nodes=30, n_timesteps=100)
print(f"Data shape: {data.shape}")


## 3. Generate and Sparsify Matrix


In [ ]:
sim_matrix = PearsonCorrelationGenerator().generate(data)
pipeline = MatrixConstructionPipeline(
    sparsifiers=[TopKRowSparsifier(k_per_node=3)],
    normalizers=[RowL1Normalizer()]
)
adj_matrix = pipeline.run(sim_matrix)
plot_adjacency_matrix_heatmap(adj_matrix, title="Sparsified Adjacency")


## 4. Node Batching (New Feature)
We'll use the `KHopBatcher` to group nodes into small neighborhoods. This is essential for mini-batch GNN training.


In [ ]:
batcher = KHopBatcher(k=1, max_batch_size=8)
batches = batcher.batch(adj_matrix)

print(f"Total batches: {len(batches)}")
for i, b in enumerate(batches[:5]):
    print(f"Batch {i}: {b}")


## 5. Summary Metrics


In [ ]:
from src.utils.graph_metrics import calculate_graph_metrics
metrics = calculate_graph_metrics(adj_matrix, directed=True)
for k, v in metrics.items():
    print(f"{k:25}: {v}")
